<a href="https://colab.research.google.com/github/Millrjess/Millrjess/blob/main/Protest_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Predicting Social Unrest Using Lagged Economic Indicators
## A Time-Series Machine Learning Approach

### Project Impact
- **F1 Score:** 0.88  
- **Recall:** 0.99  
- **Outcome:** Early warning signal for elevated and high-risk social unrest conditions.

---



## Introduction

This project builds a forward-looking machine learning system designed to predict social unrest using lagged economic indicators.  
The objective is not retrospective pattern matching, but **future risk detection**, ensuring real-world applicability for policymakers, researchers, and security analysts.



## Data Engineering

### The Pressure Cooker Effect

Social unrest rarely erupts instantly. Economic stress accumulates over time, creating a **pressure cooker effect**.

- Rising unemployment reduces household stability.
- Declining real income erodes purchasing power.
- Delayed impacts (≈180 days) reflect savings depletion, debt accumulation, and psychological stress.

These lagged variables capture *structural tension*, not momentary shocks.


In [ ]:

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("protest_data.csv")

# Ensure chronological order
df = df.sort_values("date")

# Create 180-day lag features
df["unemployment_lag_180"] = df["unemployment_rate"].shift(180)
df["income_lag_180"] = df["real_income"].shift(180)

df = df.dropna()



## Model Training with Time-Series Validation

Traditional random splits cause data leakage in temporal systems.  
We use **TimeSeriesSplit** to ensure models are trained strictly on past data.


In [ ]:

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df[["unemployment_lag_180", "income_lag_180"]]
y = df["unrest_event"]

tscv = TimeSeriesSplit(n_splits=5)

model = GradientBoostingClassifier(random_state=42)

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))



## Advanced Optimization

We apply **RandomizedSearchCV** to optimize model depth, learning rate, and ensemble size.


In [ ]:

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=tscv,
    scoring="f1",
    random_state=42
)

search.fit(X, y)
best_model = search.best_estimator_

print("Best Parameters:", search.best_params_)



## Model Interpretability (XAI)

To ensure transparency, we apply **SHAP** to explain how 180-day lagged variables influence predictions.


In [ ]:

import shap

explainer = shap.Explainer(best_model, X)
shap_values = explainer(X)

shap.summary_plot(shap_values, X)



**Insight:**  
The 180-day unemployment lag consistently shows the highest contribution to unrest probability, confirming the pressure cooker hypothesis.



## Evaluation Visualizations


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

y_prob = best_model.predict_proba(X_test)[:,1]

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()



*Data Insight:*  
High recall confirms the model's effectiveness as an early warning system rather than a precision-only classifier.



## Deployment Ready Interface


In [ ]:

class UnrestPredictor:
    def __init__(self, model):
        self.model = model

    def predict(self, city_data: pd.DataFrame):
        prob = self.model.predict_proba(city_data)[0][1]
        score = int(prob * 100)

        if score < 35:
            category = "Low"
        elif score < 65:
            category = "Elevated"
        else:
            category = "High"

        return {"Risk Score": score, "Category": category}



## Conclusion

This project demonstrates how **time-aware machine learning**, economic theory, and explainable AI combine to produce a deployable early warning system.

The methodology is generalizable to financial instability, migration risk, and geopolitical forecasting.
